# Semantic Recruitment Matcher — Pipeline hoàn chỉnh
**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5 → 6

In [16]:
# ============================================================
# CELL 1: Load datasets từ HuggingFace
# ============================================================
from datasets import load_dataset

jd_data = load_dataset(
    "lang-uk/recruitment-dataset-job-descriptions-english",
    split="train[:500]"
)
cv_data = load_dataset(
    "lang-uk/recruitment-dataset-candidate-profiles-english",
    split="train[:300]"
)

print(f"JD columns: {jd_data.column_names}")
print(f"CV columns: {cv_data.column_names}")
print(f"Loaded: {len(jd_data)} JDs, {len(cv_data)} CVs")

JD columns: ['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', '__index_level_0__']
CV columns: ['Position', 'Moreinfo', 'Looking For', 'Highlights', 'Primary Keyword', 'English Level', 'Experience Years', 'CV', 'CV_lang', 'id', '__index_level_0__']
Loaded: 500 JDs, 300 CVs


In [17]:
# ============================================================
# CELL 2: Clean data + chuẩn bị text để embed
# ============================================================
import re

def clean_text(text):
    """Xử lý null, khoảng trắng thừa, ký tự đặc biệt"""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def truncate_text(text, max_length=500):
    """Cắt tại ranh giới từ, không đứt giữa chữ"""
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] if last_space > 0 else truncated

def truncate_to_bytes(text, max_bytes):
    """Cắt chuỗi sao cho độ dài UTF-8 bytes <= max_bytes (fix multi-byte Unicode)"""
    encoded = text.encode('utf-8')
    if len(encoded) <= max_bytes:
        return text
    return encoded[:max_bytes].decode('utf-8', errors='ignore')

# --- Clean JD ---
cleaned_jd = []
skipped_jd = 0
for i, item in enumerate(jd_data):
    description = clean_text(item.get("Long Description"))
    if len(description) < 20:   # bỏ qua JD rỗng
        skipped_jd += 1
        continue
    position = clean_text(item.get("Position")) or "Unknown Position"
    keyword  = clean_text(item.get("Primary Keyword")) or ""
    company  = clean_text(item.get("Company Name")) or "Unknown Company"
    exp_years = clean_text(str(item.get("Exp Years") or ""))

    # Ghép text để embed: position + keyword + description
    text_for_embed = truncate_text(f"{position}. {keyword}. {description}")

    cleaned_jd.append({
        "id":           i + 1,
        "position":     truncate_to_bytes(position, 300),
        "description":  truncate_to_bytes(description, 2000),
        "company":      truncate_to_bytes(company, 200),
        "keyword":      truncate_to_bytes(keyword, 300),
        "exp_years":    truncate_to_bytes(exp_years, 50),
        "text_for_embed": text_for_embed
    })

# --- Clean CV ---
cleaned_cv = []
skipped_cv = 0
for i, item in enumerate(cv_data):
    cv_text = clean_text(item.get("CV"))
    if len(cv_text) < 20:       # bỏ qua CV rỗng
        skipped_cv += 1
        continue
    position    = clean_text(item.get("Position")) or "Unknown Position"
    highlights  = clean_text(item.get("Highlights")) or ""
    keyword     = clean_text(item.get("Primary Keyword")) or ""
    exp_years   = clean_text(str(item.get("Experience Years") or ""))
    looking_for = clean_text(item.get("Looking For")) or ""

    # Ghép text để embed: position + highlights + cv_text
    text_for_embed = truncate_text(f"{position}. {highlights}. {cv_text}")

    cleaned_cv.append({
        "id":           i + 1,
        "position":     truncate_to_bytes(position, 300),
        "cv_text":      truncate_to_bytes(cv_text, 2000),
        "highlights":   truncate_to_bytes(highlights, 1000),
        "keyword":      truncate_to_bytes(keyword, 300),
        "exp_years":    truncate_to_bytes(exp_years, 50),
        "looking_for":  truncate_to_bytes(looking_for, 500),
        "text_for_embed": text_for_embed
    })

print(f"JD: giữ {len(cleaned_jd)}, bỏ {skipped_jd} (rỗng)")
print(f"CV: giữ {len(cleaned_cv)}, bỏ {skipped_cv} (rỗng)")

JD: giữ 500, bỏ 0 (rỗng)
CV: giữ 300, bỏ 0 (rỗng)


In [18]:
# ============================================================
# CELL 3: Load model và embed toàn bộ data
# ============================================================
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Đang embed JDs...")
jd_texts   = [item["text_for_embed"] for item in cleaned_jd]
jd_vectors = model.encode(jd_texts, show_progress_bar=True, batch_size=64)

print("\nĐang embed CVs...")
cv_texts   = [item["text_for_embed"] for item in cleaned_cv]
cv_vectors = model.encode(cv_texts, show_progress_bar=True, batch_size=64)

print(f"\nJD vectors shape: {jd_vectors.shape}")  # (N, 384)
print(f"CV vectors shape: {cv_vectors.shape}")   # (N, 384)

Đang embed JDs...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Đang embed CVs...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


JD vectors shape: (500, 384)
CV vectors shape: (300, 384)


In [19]:
# ============================================================
# CELL 4: Drop collections cũ + Tạo lại với schema đầy đủ
# ============================================================
from pymilvus import (
    connections, utility, Collection,
    FieldSchema, CollectionSchema, DataType
)

connections.connect(host="localhost", port="19530")

# --- Drop collections cũ ---
for name in ["cvs", "job_descriptions"]:
    if utility.has_collection(name):
        Collection(name).drop()
        print(f"Dropped: {name}")

# --- Schema cho job_descriptions ---
jd_schema = CollectionSchema(
    fields=[
        FieldSchema(name="id",          dtype=DataType.INT64,        is_primary=True, auto_id=False),
        FieldSchema(name="vector",      dtype=DataType.FLOAT_VECTOR, dim=384),
        FieldSchema(name="position",    dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="description", dtype=DataType.VARCHAR,       max_length=2000),
        FieldSchema(name="company",     dtype=DataType.VARCHAR,       max_length=200),
        FieldSchema(name="keyword",     dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="exp_years",   dtype=DataType.VARCHAR,       max_length=50),
    ],
    description="Job descriptions",
    enable_dynamic_field=False
)

# --- Schema cho cvs ---
cv_schema = CollectionSchema(
    fields=[
        FieldSchema(name="id",          dtype=DataType.INT64,        is_primary=True, auto_id=False),
        FieldSchema(name="vector",      dtype=DataType.FLOAT_VECTOR, dim=384),
        FieldSchema(name="position",    dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="cv_text",     dtype=DataType.VARCHAR,       max_length=2000),
        FieldSchema(name="highlights",  dtype=DataType.VARCHAR,       max_length=1000),
        FieldSchema(name="keyword",     dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="exp_years",   dtype=DataType.VARCHAR,       max_length=50),
        FieldSchema(name="looking_for", dtype=DataType.VARCHAR,       max_length=500),
    ],
    description="Candidate CVs",
    enable_dynamic_field=False
)

jd_col = Collection(name="job_descriptions", schema=jd_schema)
cv_col = Collection(name="cvs",              schema=cv_schema)
print("Tạo collections mới thành công")

Dropped: cvs
Dropped: job_descriptions
Tạo collections mới thành công


In [20]:
# ============================================================
# CELL 5: Insert data vào Milvus
# ============================================================

# --- Insert JDs ---
jd_insert = [
    {
        "id": int(item["id"]),
        "vector": vec.tolist(),
        "position": item["position"],
        "description": item["description"],
        "company": item["company"],
        "keyword": item["keyword"],
        "exp_years": item["exp_years"],
    }
    for item, vec in zip(cleaned_jd, jd_vectors)
]
jd_col.insert(jd_insert)
print(f"Inserted {len(cleaned_jd)} JDs")

# --- Insert CVs ---
cv_insert = [
    {
        "id": int(item["id"]),
        "vector": vec.tolist(),
        "position": item["position"],
        "cv_text": item["cv_text"],
        "highlights": item["highlights"],
        "keyword": item["keyword"],
        "exp_years": item["exp_years"],
        "looking_for": item["looking_for"],
    }
    for item, vec in zip(cleaned_cv, cv_vectors)
]
cv_col.insert(cv_insert)
print(f"Inserted {len(cleaned_cv)} CVs")

# --- Tạo index và load vào RAM ---
index_params = {
    "index_type": "AUTOINDEX",
    "metric_type": "COSINE",
    "params": {}
}

jd_col.create_index(field_name="vector", index_params=index_params)
cv_col.create_index(field_name="vector", index_params=index_params)

jd_col.load()
cv_col.load()
print("Index + load hoàn tất")

Inserted 500 JDs
Inserted 300 CVs
Index + load hoàn tất


In [21]:
# ============================================================
# CELL 6: Test search để verify pipeline hoạt động
# ============================================================

query = "Python backend developer with Django and REST API experience"
query_vector = model.encode([query])

results = cv_col.search(
    data=query_vector.tolist(),
    anns_field="vector",
    param={"metric_type": "COSINE"},
    limit=5,
    output_fields=["position", "keyword", "cv_text"]
)

print(f"Query: '{query}'\n")
print("=== Top 5 CV phù hợp ===")
for i, hit in enumerate(results[0]):
    print(f"\n#{i+1} | Score: {hit.score:.4f}")
    print(f"   Position: {hit.entity.get('position')}")
    print(f"   Keyword:  {hit.entity.get('keyword')}")
    print(f"   CV:       {hit.entity.get('cv_text')[:100]}...")

Query: 'Python backend developer with Django and REST API experience'

=== Top 5 CV phù hợp ===

#1 | Score: 0.3680
   Position: 1C developer
   Keyword:  Flutter
   CV:       1 am an 1C developer. I deployed an 1C to typographical factory in Ukraine. Also I deploy 1C to othe...

#2 | Score: 0.3618
   Position: 1С Developer
   Keyword:  Other
   CV:       1C 8.2, 8.3 programming, reports, processing, SKD. UPP, custom configurations, managed and usual for...

#3 | Score: 0.2884
   Position: 1c Developer
   Keyword:  Other
   CV:       Worked on a mobile application for tracking trips...

#4 | Score: 0.2502
   Position: 2d/3d concept artist
   Keyword:  Other
   CV:       Full development pipeline from quick concepts to Concept models in a 3D editor or engine CG generali...

#5 | Score: 0.2345
   Position: 2D Animator / Motion Designer
   Keyword:  Artist
   CV:       Created 4 explainer videos from scratch. I have 1+ year of experience working in Adobe After Effects...
